In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/mkashifn/nbaiot-dataset/7.gafgyt.combo.csv
/kaggle/input/datasets/mkashifn/nbaiot-dataset/9.gafgyt.combo.csv
/kaggle/input/datasets/mkashifn/nbaiot-dataset/5.gafgyt.combo.csv
/kaggle/input/datasets/mkashifn/nbaiot-dataset/1.mirai.udp.csv
/kaggle/input/datasets/mkashifn/nbaiot-dataset/4.gafgyt.udp.csv
/kaggle/input/datasets/mkashifn/nbaiot-dataset/6.gafgyt.udp.csv
/kaggle/input/datasets/mkashifn/nbaiot-dataset/6.gafgyt.junk.csv
/kaggle/input/datasets/mkashifn/nbaiot-dataset/data_summary.csv
/kaggle/input/datasets/mkashifn/nbaiot-dataset/5.gafgyt.udp.csv
/kaggle/input/datasets/mkashifn/nbaiot-dataset/9.gafgyt.junk.csv
/kaggle/input/datasets/mkashifn/nbaiot-dataset/9.mirai.scan.csv
/kaggle/input/datasets/mkashifn/nbaiot-dataset/1.benign.csv
/kaggle/input/datasets/mkashifn/nbaiot-dataset/2.mirai.udpplain.csv
/kaggle/input/datasets/mkashifn/nbaiot-dataset/3.gafgyt.combo.csv
/kaggle/input/datasets/mkashifn/nbaiot-dataset/4.gafgyt.combo.csv
/kaggle/input/datasets/mkashi

In [3]:
# ===== CELL 1: Imports and Configuration =====
 
import os, gc, time, random, warnings, json, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from pathlib import Path
 
warnings.filterwarnings("ignore")
 
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    HAVE_PLOT = True
except ImportError:
    HAVE_PLOT = False
 
try:
    from scipy import stats as sp_stats
    HAVE_SCIPY = True
except ImportError:
    HAVE_SCIPY = False
    print("scipy not found — statistical tests skipped")
 
def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    if torch.cuda.is_available():
        torch.cuda.synchronize()
 
# ── Configuration ─────────────────────────────────────────────────────────────
CFG = {
    "DATA_PATH":         "/kaggle/input/nbaiot-dataset",
    "SAMPLE_PER_DEVICE": 50_000,
    "FL_ROUNDS":         30,
    "LOCAL_EPOCHS":      2,
    "BATCH_SIZE":        512,
    "LR":                0.001,
    "GM_MAX_ITERS":      100,
    "GM_TOL":            1e-5,
    "GM_NU":             1e-6,
    "TRAIN_RATIO":       0.70,
    "VAL_RATIO":         0.15,
    "TEST_RATIO":        0.15,
    "SEEDS":             [42, 123, 456, 789, 1011, 1415, 1617, 1819, 2021],
    "BYZ_RATIO":         0.40,    # all runs at 40% Byzantine (primary condition)
    "OUT_DIR":           "/kaggle/working",
    "CSV_NAME":          "sensitivity_session1.csv",
    "CHECKPOINT_EVERY":  9,       # save after every complete sweep over 9 seeds
}
 
# Experiment parameter grids
EXP1_PD_VALS    = [0.30, 0.40, 0.50, 0.60, 0.70]   # 45 runs, ~2.44h
EXP2_PFP_VALS   = [0.04, 0.08, 0.12, 0.15]          # 36 runs, ~1.95h
EXP3_PRIOR_VALS = [0.20, 0.40, 0.50]                 # 27 runs, ~1.46h
# Exp4: BST-FedAvg ablation                           #  9 runs, ~0.49h
# Total: 117 runs × ~195s = ~6.33h
 
DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FEAT_COLS = None
INPUT_DIM = None
 
# ── Precomputed baseline reference values (from bst_gm_results.csv) ─────────
# These are per-seed ASR values at 40% Byzantine, matched to SEEDS order.
# Used only for statistical tests in Cell 13.
# Verified: same seeds, same byz_ids, same preprocessing (seed=42 split).
TRUST_GM_40PCT_PER_SEED = {   # Trust-GM per-seed ASR at 40% Byzantine
    42:   0.256737, 123: 0.019376, 456: 0.087194, 789: 0.080735,
    1011: 0.050056, 1415: 0.067817, 1617: 0.048441, 1819: 0.016147, 2021: 0.029065,
}
GM_40PCT_PER_SEED = {         # Plain GM per-seed ASR at 40% Byzantine
    42:   7.844214, 123: 0.134020, 456: 0.322940, 789: 2.877396,
    1011: 0.406904, 1415: 4.881239, 1617: 0.581292, 1819: 0.295490, 2021: 0.348775,
}
BST_GM_40PCT_PER_SEED = {     # BST-GM (p_d=0.70) per-seed ASR at 40% Byzantine
    42:   0.022606, 123: 0.024221, 456: 0.029065, 789: 0.027450,
    1011: 0.032294, 1415: 0.020991, 1617: 0.027450, 1819: 0.022606, 2021: 0.017762,
}
 
n_runs_session1 = (len(EXP1_PD_VALS) + len(EXP2_PFP_VALS) +
                   len(EXP3_PRIOR_VALS) + 1) * len(CFG["SEEDS"])
 
print(f"Device     : {DEVICE}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}  : {torch.cuda.get_device_name(i)}")
print(f"Total runs : {n_runs_session1}")
print(f"Est. time  : {n_runs_session1 * 195 / 3600:.2f}h  (measured 195s/run)")
print(f"Safe budget: 10.0h  [{'FITS ✓' if n_runs_session1 * 195 / 3600 <= 10.0 else 'TOO LONG ✗'}]")
print("✓ Cell 1 complete")

Device     : cuda
  GPU 0  : Tesla T4
  GPU 1  : Tesla T4
Total runs : 117
Est. time  : 6.34h  (measured 195s/run)
Safe budget: 10.0h  [FITS ✓]
✓ Cell 1 complete


In [4]:
# ===== CELL 2: Dataset Loading =====
 
_DEVICE_NAMES = {
    "1":"Danmini_Doorbell",     "2":"Ecobee_Thermostat",
    "3":"Ennio_Doorbell",       "4":"Philips_B120N10_Baby_Monitor",
    "5":"Provision_PT_737E_Camera","6":"Provision_PT_838_Camera",
    "7":"Samsung_SNH_1011_N_Webcam","8":"SimpleHome_XCS7_1002_Camera",
    "9":"SimpleHome_XCS7_1003_Camera",
}
_DEVICE_IDS = ["1","2","3","4","5","6","7","8","9"]
_ATTACK_SUFFIXES = [
    "benign","gafgyt.combo","gafgyt.junk","gafgyt.scan","gafgyt.tcp","gafgyt.udp",
    "mirai.ack","mirai.scan","mirai.syn","mirai.udp","mirai.udpplain",
]
_CANDIDATES = [
    "/kaggle/input/nbaiot-dataset",
    "/kaggle/input/mkashifn-nbaiot-dataset",
    "/kaggle/input/N-BaIoT","/kaggle/input/nbaiot","/kaggle/input/n-baiot",
]
 
 
def _is_nbaiot_root(p):
    p = Path(p)
    return p.is_dir() and any((p/f).exists() for f in ["1.benign.csv","2.benign.csv"])
 
 
def _resolve_path(cfg_path):
    if _is_nbaiot_root(cfg_path): return cfg_path
    for c in _CANDIDATES:
        if _is_nbaiot_root(c): return c
    ki = Path("/kaggle/input")
    if ki.exists():
        for d in sorted(ki.rglob("*")):
            if d.is_dir() and _is_nbaiot_root(d): return str(d)
    raise RuntimeError("N-BaIoT not found. Add Data → nbaiot-dataset by mkashifn.")
 
 
def load_nbaiot(base_path, sample_per_device=None, seed=42):
    rng  = np.random.RandomState(seed)
    base = Path(_resolve_path(base_path))
    print(f"Root: {base}")
    all_dfs = []
    for did in _DEVICE_IDS:
        idx = int(did) - 1
        parts = []
        for suf in _ATTACK_SUFFIXES:
            fp = base / f"{did}.{suf}.csv"
            if not fp.exists(): continue
            try:
                df_p = pd.read_csv(fp, header=0)
            except: continue
            if df_p.shape[1] != 115: continue
            df_p["label"]     = 0 if suf == "benign" else 1
            df_p["device_id"] = idx
            parts.append(df_p)
        if not parts: continue
        dev = pd.concat(parts, ignore_index=True)
        del parts; gc.collect()
        if sample_per_device and len(dev) > sample_per_device:
            ben = dev[dev["label"]==0]; att = dev[dev["label"]==1]
            rb = len(ben)/len(dev)
            nb = max(1, int(sample_per_device*rb)); na = sample_per_device - nb
            sb = ben.sample(n=min(nb,len(ben)), random_state=rng.randint(0,2**31))
            sa = att.sample(n=min(na,len(att)), random_state=rng.randint(0,2**31))
            dev = pd.concat([sb, sa], ignore_index=True)
            del ben, att, sb, sa; gc.collect()
        pct = (dev["label"]==0).sum()/len(dev)*100
        print(f"  Device {did} ({_DEVICE_NAMES[did][:24]:24s}): {len(dev):>7,}  [{pct:.1f}%]")
        all_dfs.append(dev); del dev; gc.collect()
    df  = pd.concat(all_dfs, ignore_index=True); del all_dfs; gc.collect()
    fc  = [c for c in df.columns if c not in ("label","device_id")]
    df[fc] = df[fc].apply(pd.to_numeric, errors="coerce")
    df = df.dropna().reset_index(drop=True)
    print(f"Total: {len(df):,}  ({(df['label']==0).sum()/len(df)*100:.1f}% benign)")
    return df
 
 
print("Loading N-BaIoT …")
t0 = time.time()
RAW_DF = load_nbaiot(CFG["DATA_PATH"], sample_per_device=CFG["SAMPLE_PER_DEVICE"], seed=42)
FEAT_COLS = [c for c in RAW_DF.columns if c not in ("label","device_id")]
INPUT_DIM = len(FEAT_COLS)
assert INPUT_DIM == 115, f"Expected 115 features, got {INPUT_DIM}"
print(f"Features: {INPUT_DIM}  |  Load: {(time.time()-t0)/60:.1f} min")
print("✓ Cell 2 complete")

Loading N-BaIoT …
Root: /kaggle/input/datasets/mkashifn/nbaiot-dataset
  Device 1 (Danmini_Doorbell        ):  50,000  [4.9%]
  Device 2 (Ecobee_Thermostat       ):  50,000  [1.6%]
  Device 3 (Ennio_Doorbell          ):  50,000  [11.0%]
  Device 4 (Philips_B120N10_Baby_Mon):  50,000  [16.0%]
  Device 5 (Provision_PT_737E_Camera):  50,000  [7.5%]
  Device 6 (Provision_PT_838_Camera ):  50,000  [11.8%]
  Device 7 (Samsung_SNH_1011_N_Webca):  50,000  [13.9%]
  Device 8 (SimpleHome_XCS7_1002_Cam):  50,000  [5.4%]
  Device 9 (SimpleHome_XCS7_1003_Cam):  50,000  [2.3%]
Total: 450,000  (8.2% benign)
Features: 115  |  Load: 3.7 min
✓ Cell 2 complete


In [5]:
# ===== CELL 3: Preprocessing (seed=42, identical to prior experiments) =====
 
def split_per_device(df, seed=42):
    """Stratified 70/15/15 per device. Fixed seed=42 — same split as all prior runs."""
    tr_l, va_l, te_l = [], [], []
    for dev in sorted(df["device_id"].unique()):
        sub = df[df["device_id"]==dev]
        try:
            tv, te = train_test_split(sub, test_size=CFG["TEST_RATIO"],
                                       random_state=seed, stratify=sub["label"])
            adj = CFG["VAL_RATIO"]/(CFG["TRAIN_RATIO"]+CFG["VAL_RATIO"])
            tr, va = train_test_split(tv, test_size=adj,
                                       random_state=seed, stratify=tv["label"])
        except:
            tv, te = train_test_split(sub, test_size=CFG["TEST_RATIO"], random_state=seed)
            adj = CFG["VAL_RATIO"]/(CFG["TRAIN_RATIO"]+CFG["VAL_RATIO"])
            tr, va = train_test_split(tv, test_size=adj, random_state=seed)
        tr_l.append(tr); va_l.append(va); te_l.append(te)
    return (pd.concat(tr_l,ignore_index=True), pd.concat(va_l,ignore_index=True),
            pd.concat(te_l,ignore_index=True))
 
 
def scale_no_leakage(tr, va, te, fc):
    """Fit StandardScaler on TRAIN only — zero leakage."""
    sc = StandardScaler()
    sc.fit(tr[fc].values.astype(np.float64))
    ts, vs, tes = tr.copy(), va.copy(), te.copy()
    for df_out, df_in in [(ts,tr),(vs,va),(tes,te)]:
        df_out[fc] = sc.transform(df_in[fc].values.astype(np.float64)).astype(np.float32)
    return ts, vs, tes, sc
 
 
def compute_client_class_weights(train_df):
    """Returns {device_id: tensor([w_benign, w_attack])}. Capped at 50."""
    weights = {}
    for dev in sorted(train_df["device_id"].unique()):
        sub   = train_df[train_df["device_id"]==dev]
        n_ben = int((sub["label"]==0).sum())
        n_att = int((sub["label"]==1).sum())
        if n_ben == 0 or n_att == 0:
            weights[dev] = torch.tensor([1.0, 1.0], dtype=torch.float32)
        else:
            weights[dev] = torch.tensor([min(n_att/n_ben, 50.0), 1.0], dtype=torch.float32)
    return weights
 
 
print("Splitting and scaling (seed=42) …")
TRAIN_DF, VAL_DF, TEST_DF = split_per_device(RAW_DF, seed=42)
TRAIN_DF, VAL_DF, TEST_DF, SCALER = scale_no_leakage(TRAIN_DF, VAL_DF, TEST_DF, FEAT_COLS)
del RAW_DF; gc.collect()
CLIENT_CLASS_WEIGHTS = compute_client_class_weights(TRAIN_DF)
TEST_DS = TensorDataset(
    torch.tensor(TEST_DF[FEAT_COLS].values, dtype=torch.float32),
    torch.tensor(TEST_DF["label"].values,   dtype=torch.long),
)
TEST_LOADER = DataLoader(TEST_DS, batch_size=2048, shuffle=False,
                          num_workers=0, pin_memory=(DEVICE.type=="cuda"))
print(f"Test: {len(TEST_DS):,} samples")
print("✓ Cell 3 complete")

Splitting and scaling (seed=42) …
Test: 67,500 samples
✓ Cell 3 complete


In [6]:
# ===== CELL 4: Model =====
 
class IDSModel(nn.Module):
    """3-layer MLP: 115→128→64→2 (ReLU, no BatchNorm). 23,234 params."""
    def __init__(self, input_dim=None, h1=128, h2=64, n_classes=2):
        super().__init__()
        if input_dim is None: input_dim = INPUT_DIM
        self.net = nn.Sequential(
            nn.Linear(input_dim, h1), nn.ReLU(),
            nn.Linear(h1, h2),        nn.ReLU(),
            nn.Linear(h2, n_classes),
        )
    def forward(self, x): return self.net(x)
 
 
_t = IDSModel()
assert sum(v.numel() for v in _t.state_dict().values()) == 23_234
assert len(list(_t.buffers())) == 0, "Unexpected BatchNorm buffers"
del _t
print("IDSModel: 23,234 params, no BatchNorm ✓")
print("✓ Cell 4 complete")
 

IDSModel: 23,234 params, no BatchNorm ✓
✓ Cell 4 complete


In [7]:
# ===== CELL 5: FL Utilities =====
 
def sd_to_np(sd):
    return np.concatenate(
        [v.detach().cpu().numpy().flatten() for v in sd.values()]
    ).astype(np.float64)
 
def np_to_sd(vec, template):
    out = {}; idx = 0
    for k, v in template.items():
        n = v.numel()
        out[k] = torch.from_numpy(vec[idx:idx+n].reshape(v.shape).astype(np.float32))
        idx += n
    return out
 
def create_client_loaders(train_df, feat_cols, batch_size, seed):
    loaders, sizes = [], []
    for dev in sorted(train_df["device_id"].unique()):
        sub = train_df[train_df["device_id"]==dev]
        ds  = TensorDataset(
            torch.tensor(sub[feat_cols].values, dtype=torch.float32),
            torch.tensor(sub["label"].values,   dtype=torch.long),
        )
        gen = torch.Generator()
        gen.manual_seed(int(seed) + int(dev))
        dl  = DataLoader(ds, batch_size=batch_size, shuffle=True, generator=gen,
                         drop_last=False, num_workers=0, pin_memory=(DEVICE.type=="cuda"))
        loaders.append(dl); sizes.append(len(ds))
    return loaders, sizes
 
def train_local(global_sd, loader, epochs, lr, device, flip_labels=False, class_weights=None):
    model = IDSModel(input_dim=INPUT_DIM).to(device)
    model.load_state_dict({k: v.to(device) for k,v in global_sd.items()})
    model.train()
    opt  = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss(
        weight=class_weights.to(device) if class_weights is not None else None
    )
    for _ in range(epochs):
        for Xb, yb in loader:
            Xb = Xb.to(device); yb = yb.to(device)
            if flip_labels: yb = 1 - yb
            opt.zero_grad(); crit(model(Xb), yb).backward(); opt.step()
    return {k: v.cpu() for k,v in model.state_dict().items()}
 
def evaluate_global(global_sd, test_loader, device):
    model = IDSModel(input_dim=INPUT_DIM).to(device)
    model.load_state_dict({k: v.to(device) for k,v in global_sd.items()})
    model.eval()
    all_p, all_y = [], []
    with torch.no_grad():
        for Xb, yb in test_loader:
            all_p.extend(model(Xb.to(device)).argmax(dim=1).cpu().numpy())
            all_y.extend(yb.numpy())
    cm             = confusion_matrix(all_y, all_p, labels=[0,1])
    tn, fp, fn, tp = cm.ravel()
    asr = (fn/(fn+tp))*100 if (fn+tp)>0 else 0.0
    fpr = (fp/(fp+tn))*100 if (fp+tn)>0 else 0.0
    return {"asr": asr, "fpr": fpr,
            "balanced_accuracy": ((100-asr)+(100-fpr))/2.0,
            "tn":int(tn),"fp":int(fp),"fn":int(fn),"tp":int(tp)}
 
def init_global_model(seed):
    set_all_seeds(seed)
    return {k: v.cpu() for k,v in IDSModel(input_dim=INPUT_DIM).state_dict().items()}
 
print("✓ Cell 5 complete")

✓ Cell 5 complete


In [8]:
# ===== CELL 6: Aggregation Primitives =====
 
def _compute_and_clip_deltas(client_sds, global_sd):
    deltas   = [{k: csd[k].float()-global_sd[k].float() for k in global_sd}
                for csd in client_sds]
    vecs     = [sd_to_np(d) for d in deltas]
    norms    = np.array([np.linalg.norm(v) for v in vecs])
    clip_val = float(np.median(norms))
    clipped  = []
    for v, n in zip(vecs, norms):
        clipped.append(v*(clip_val/n) if clip_val > 0.0 and n > clip_val else v.copy())
    return clipped, deltas
 
def _weiszfeld(vecs, tau, max_iters, tol, nu):
    tau_sum = tau.sum()
    if tau_sum < 1e-12:
        tau = np.ones(len(vecs), dtype=np.float64); tau_sum = float(len(vecs))
    w = np.sum([t*v for t,v in zip(tau,vecs)], axis=0) / tau_sum
    for _ in range(max_iters):
        dists = np.maximum(np.array([np.linalg.norm(w-v) for v in vecs]), nu)
        w_new = (np.sum([t*v/d for t,v,d in zip(tau,vecs,dists)], axis=0)
                 / np.sum(tau/dists))
        if np.linalg.norm(w_new - w) < tol: break
        w = w_new
    return w
 
def geometric_median(client_sds, global_sd):
    clipped, deltas = _compute_and_clip_deltas(client_sds, global_sd)
    tau = np.ones(len(clipped), dtype=np.float64)
    w   = _weiszfeld(clipped, tau, CFG["GM_MAX_ITERS"], CFG["GM_TOL"], CFG["GM_NU"])
    agg_d = np_to_sd(w, deltas[0])
    return {k: global_sd[k].float()+agg_d[k] for k in global_sd}, w
 
print("✓ Cell 6 complete")

✓ Cell 6 complete


In [9]:
# ===== CELL 7: BST-GM Trust Engine =====
 
class BayesianSequentialTrust:
    """
    Bayesian Sequential Trust (BST) for FL honeypot defense.
 
    [FIX-1] Prior and observations are SEPARATE:
      - _log_prior_odds: FIXED constant, never decayed.
      - _accum_obs[i]: ONLY the accumulated observation log-likelihood ratio.
        This is decayed and updated per round.
      - get_trust(): total LLR = fixed_prior + accumulated_obs.
      - Correct Bayesian update: prior stays fixed as background belief;
        decay applies only to the observation evidence (recent evidence
        matters more than old evidence when decay < 1.0).
    """
    def __init__(self, p_d, p_fp, prior_byz=0.40, decay=1.0, tau_min=0.01, seed=42):
        assert 0 < p_d < 1 and 0 < p_fp < 1
        assert 0 < prior_byz < 1 and 0 < decay <= 1.0 and 0 < tau_min < 1
        self.tau_min         = tau_min
        self.decay           = decay
        self.rng             = np.random.RandomState(seed)
        self._p_d            = p_d
        self._p_fp           = p_fp
        # [FIX-1] Prior is FIXED — stored once, never modified
        self._log_prior_odds = float(math.log(prior_byz / (1.0 - prior_byz)))
        # Precomputed LLR values for efficiency
        self._llr_det        = float(math.log(p_d / p_fp))
        self._llr_nodet      = float(math.log((1.0 - p_d) / (1.0 - p_fp)))
        # [FIX-1] Only observation evidence accumulated here (NOT the prior)
        self._accum_obs: dict = {}
 
    def simulate_round(self, client_id, is_byzantine):
        prob     = self._p_d if is_byzantine else self._p_fp
        detected = bool(self.rng.random() < prob)
        self._update_obs(client_id, detected)
        return detected
 
    def _update_obs(self, client_id, detected):
        """
        [FIX-1] Update OBSERVATION EVIDENCE only (prior not touched).
        1. Initialize observation accumulator at 0.0 if first round.
        2. Decay existing evidence.
        3. Add new observation's LLR.
        4. Clip to [-50, 50] to prevent sigmoid overflow.
        """
        if client_id not in self._accum_obs:
            self._accum_obs[client_id] = 0.0   # no prior here — prior is separate
        self._accum_obs[client_id] *= self.decay
        self._accum_obs[client_id] += self._llr_det if detected else self._llr_nodet
        self._accum_obs[client_id] = float(np.clip(self._accum_obs[client_id], -50.0, 50.0))
 
    def get_trust(self, client_id):
        """
        [FIX-1] Total LLR = FIXED prior + accumulated observations.
        Trust = sigmoid(-total_LLR) = P(Benign | observations).
        """
        obs  = self._accum_obs.get(client_id, 0.0)
        llr  = float(np.clip(self._log_prior_odds + obs, -50.0, 50.0))
        p_byz = 1.0 / (1.0 + math.exp(-llr))
        return max(self.tau_min, 1.0 - p_byz)
 
    def get_all_trusts(self, n):
        return np.array([self.get_trust(i) for i in range(n)], dtype=np.float64)
 
 
def bst_gm(client_sds, trust_scores, global_sd):
    """BST-weighted geometric median (delta-space + median clipping + Weiszfeld)."""
    clipped, deltas = _compute_and_clip_deltas(client_sds, global_sd)
    tau = np.asarray(trust_scores, dtype=np.float64)
    if tau.sum() < 1e-12: tau = np.ones(len(clipped), dtype=np.float64)
    w     = _weiszfeld(clipped, tau, CFG["GM_MAX_ITERS"], CFG["GM_TOL"], CFG["GM_NU"])
    agg_d = np_to_sd(w, deltas[0])
    return {k: global_sd[k].float()+agg_d[k] for k in global_sd}
 
 
def bst_fedavg(client_sds, trust_scores):
    """
    Ablation: Bayesian trust weights applied as a weighted average.
    No delta projection, no geometric median, no clipping.
 
    [FIX-2] Uses PURE trust weights only (τᵢ / Στᵢ), NOT tau × dataset_size.
    The original implementation used combined = tau * size, which conflated
    trust weighting with dataset-size weighting. This makes the ablation
    unclean: we cannot tell how much of the improvement comes from the
    trust signal vs. from re-weighting by dataset size.
    With pure trust weights, the ablation cleanly answers:
      "Does Bayesian trust scoring alone (without GM) outperform plain GM?"
    """
    tau   = np.asarray(trust_scores, dtype=np.float64)
    total = tau.sum()
    if total < 1e-12:
        tau   = np.ones(len(client_sds), dtype=np.float64)
        total = float(len(client_sds))
    agg = {}
    for k in client_sds[0]:
        agg[k] = sum((tau[i] / total) * client_sds[i][k].float()
                     for i in range(len(client_sds)))
    return agg
 
 
# Verify BST correctness analytically
_bst_test = BayesianSequentialTrust(p_d=0.70, p_fp=0.08, prior_byz=0.40,
                                     decay=1.0, tau_min=0.01, seed=0)
for _ in range(21): _bst_test._update_obs(0, True)   # 21 detections (E[k|30 rds])
for _ in range(9):  _bst_test._update_obs(0, False)  # 9 non-detections
_tau_byz = _bst_test.get_trust(0)
 
_bst_hon = BayesianSequentialTrust(p_d=0.70, p_fp=0.08, prior_byz=0.40,
                                    decay=1.0, tau_min=0.01, seed=1)
for _ in range(2):  _bst_hon._update_obs(0, True)   # 2 FPs (E[k|30 rds, p_fp=0.08])
for _ in range(28): _bst_hon._update_obs(0, False)  # 28 clean
_tau_hon = _bst_hon.get_trust(0)
 
print(f"BST [FIX-1] verified:")
print(f"  E[τ_byz|t=30]: {_tau_byz:.4f}  (target: 0.0100, step fn floor: 0.1500)")
print(f"  E[τ_hon|t=30]: {_tau_hon:.4f}  (target: 1.0000, step fn: 0.3210)")
del _bst_test, _bst_hon, _tau_byz, _tau_hon
print("✓ Cell 7 complete — BST-GM and BST-FedAvg defined")

BST [FIX-1] verified:
  E[τ_byz|t=30]: 0.0100  (target: 0.0100, step fn floor: 0.1500)
  E[τ_hon|t=30]: 1.0000  (target: 1.0000, step fn: 0.3210)
✓ Cell 7 complete — BST-GM and BST-FedAvg defined


In [10]:
# ===== CELL 8: Experiment Runner =====
 
def run_bst_experiment(method, seed, p_d=0.70, p_fp=0.08, prior_byz=0.40,
                        bst_decay=1.0, bst_tau_min=0.01):
    """
    Run one FL experiment with configurable BST parameters.
    method ∈ {'bst_gm', 'bst_fedavg'}
    RNG isolation identical to prior experiments (seed, seed+10000, etc.)
    """
    if torch.cuda.is_available(): torch.cuda.synchronize()
    set_all_seeds(seed)
 
    client_loaders, client_sizes = create_client_loaders(
        TRAIN_DF, FEAT_COLS, CFG["BATCH_SIZE"], seed)
    n_clients = len(client_loaders)
 
    n_byz   = round(n_clients * CFG["BYZ_RATIO"])
    byz_ids = set(np.random.RandomState(seed).choice(
        n_clients, size=n_byz, replace=False).tolist())
 
    global_sd = init_global_model(seed)
 
    bst = BayesianSequentialTrust(
        p_d=p_d, p_fp=p_fp, prior_byz=prior_byz,
        decay=bst_decay, tau_min=bst_tau_min,
        seed=seed + 10_000,
    )
 
    for rnd in range(CFG["FL_ROUNDS"]):
        client_updates = []
        for cid, loader in enumerate(client_loaders):
            cw = CLIENT_CLASS_WEIGHTS.get(cid, torch.tensor([1.0, 1.0]))
            sd = train_local(global_sd, loader,
                             epochs=CFG["LOCAL_EPOCHS"], lr=CFG["LR"],
                             device=DEVICE, flip_labels=(cid in byz_ids),
                             class_weights=cw)
            client_updates.append(sd)
 
        # Honeypot simulation before aggregation
        for cid in range(n_clients):
            bst.simulate_round(cid, is_byzantine=(cid in byz_ids))
        tau = bst.get_all_trusts(n_clients)
 
        if method == "bst_gm":
            global_sd = bst_gm(client_updates, tau, global_sd)
        elif method == "bst_fedavg":
            global_sd = bst_fedavg(client_updates, tau)  # [FIX-2] pure tau weights
        else:
            raise ValueError(f"Unknown method: {method}")
 
        del client_updates
 
    return evaluate_global(global_sd, TEST_LOADER, DEVICE), sorted(byz_ids), n_byz
 
 
print("✓ Cell 8 complete — experiment runner defined")

✓ Cell 8 complete — experiment runner defined


In [11]:
# ===== CELL 9: Run All BST Experiments =====
# [FIX-3] ETA uses n_runs_session1 (correct count)
# [FIX-4] Checkpoint after each experiment block
 
print("=" * 65)
print("SESSION 1: BST-GM SENSITIVITY EXPERIMENTS")
print("=" * 65)
 
all_results = []
t_start = time.time()
run_idx = 0
_chk_path = f"{CFG['OUT_DIR']}/sensitivity_session1_checkpoint.csv"
 
 
def _save_checkpoint(label=""):
    """Save intermediate results to protect against kernel crash."""
    if all_results:
        pd.DataFrame(all_results).to_csv(_chk_path, index=False)
        if label:
            print(f"  ✓ Checkpoint [{label}]: {len(all_results)} rows → {_chk_path}")
 
 
def _log_run(run_idx, n_total, param_name, param_val, seed, metrics):
    elapsed_total = time.time() - t_start
    eta_h = (elapsed_total / run_idx) * (n_total - run_idx) / 3600
    return (f"  [{run_idx:3d}/{n_total}] {param_name}={param_val}  seed={seed:4d}  "
            f"ASR={metrics['asr']:.4f}%  FPR={metrics['fpr']:.4f}%  ETA={eta_h:.1f}h")
 
 
# ── Experiment 1: p_d sensitivity ─────────────────────────────────────────────
print(f"\n{'─'*50}")
print("EXPERIMENT 1: p_d sensitivity (CRITICAL)")
print("  Fixed: p_fp=0.08, prior_byz=0.40, byz=40%")
print("  Tests: deployability range — at what p_d does BST-GM remain robust?")
print(f"{'─'*50}")
 
for p_d_val in EXP1_PD_VALS:
    for seed in CFG["SEEDS"]:
        run_idx += 1
        t0 = time.time()
        metrics, byz_ids, n_byz = run_bst_experiment(
            method="bst_gm", seed=seed,
            p_d=p_d_val, p_fp=0.08, prior_byz=0.40,
        )
        elapsed = time.time() - t0
        all_results.append({
            "experiment": "exp1_pd", "method": "bst_gm",
            "byz_ratio": CFG["BYZ_RATIO"], "seed": seed,
            "param_name": "p_d", "param_value": p_d_val,
            "asr": metrics["asr"], "fpr": metrics["fpr"],
            "balanced_accuracy": metrics["balanced_accuracy"],
            "tn": metrics["tn"], "fp": metrics["fp"],
            "fn": metrics["fn"], "tp": metrics["tp"],
            "elapsed_s": round(elapsed, 1),
        })
        gc.collect()
        print(_log_run(run_idx, n_runs_session1, "p_d", f"{p_d_val:.2f}",
                       seed, metrics))
    _save_checkpoint(f"Exp1 p_d={p_d_val:.2f} complete")
 
print(f"\n  Exp1 complete. Printing per-p_d summary:")
_exp1 = pd.DataFrame([r for r in all_results if r["experiment"]=="exp1_pd"])
for v in EXP1_PD_VALS:
    s = _exp1[_exp1["param_value"]==v]
    print(f"    p_d={v:.2f}: ASR={s['asr'].mean():.4f}%±{s['asr'].std():.4f}%  "
          f"max={s['asr'].max():.4f}%  FPR={s['fpr'].mean():.4f}%")
 
 
# ── Experiment 2: p_fp sensitivity ────────────────────────────────────────────
print(f"\n{'─'*50}")
print("EXPERIMENT 2: p_fp sensitivity (CRITICAL)")
print("  Fixed: p_d=0.70, prior_byz=0.40, byz=40%")
print("  Tests: how does false-positive noise affect robustness and clean FPR?")
print(f"{'─'*50}")
 
for p_fp_val in EXP2_PFP_VALS:
    for seed in CFG["SEEDS"]:
        run_idx += 1
        t0 = time.time()
        metrics, byz_ids, n_byz = run_bst_experiment(
            method="bst_gm", seed=seed,
            p_d=0.70, p_fp=p_fp_val, prior_byz=0.40,
        )
        elapsed = time.time() - t0
        all_results.append({
            "experiment": "exp2_pfp", "method": "bst_gm",
            "byz_ratio": CFG["BYZ_RATIO"], "seed": seed,
            "param_name": "p_fp", "param_value": p_fp_val,
            "asr": metrics["asr"], "fpr": metrics["fpr"],
            "balanced_accuracy": metrics["balanced_accuracy"],
            "tn": metrics["tn"], "fp": metrics["fp"],
            "fn": metrics["fn"], "tp": metrics["tp"],
            "elapsed_s": round(elapsed, 1),
        })
        gc.collect()
        print(_log_run(run_idx, n_runs_session1, "p_fp", f"{p_fp_val:.2f}",
                       seed, metrics))
    _save_checkpoint(f"Exp2 p_fp={p_fp_val:.2f} complete")
 
print(f"\n  Exp2 complete. Per-p_fp summary:")
_exp2 = pd.DataFrame([r for r in all_results if r["experiment"]=="exp2_pfp"])
for v in EXP2_PFP_VALS:
    s = _exp2[_exp2["param_value"]==v]
    print(f"    p_fp={v:.2f}: ASR={s['asr'].mean():.4f}%  FPR={s['fpr'].mean():.4f}%  "
          f"max_ASR={s['asr'].max():.4f}%")
 
 
# ── Experiment 3: prior_byz sensitivity ───────────────────────────────────────
print(f"\n{'─'*50}")
print("EXPERIMENT 3: prior_byz sensitivity")
print("  Fixed: p_d=0.70, p_fp=0.08, byz=40%")
print("  0.20=uninformed  0.40=oracle assumption  0.50=max-entropy")
print(f"{'─'*50}")
 
for prior_val in EXP3_PRIOR_VALS:
    for seed in CFG["SEEDS"]:
        run_idx += 1
        t0 = time.time()
        metrics, byz_ids, n_byz = run_bst_experiment(
            method="bst_gm", seed=seed,
            p_d=0.70, p_fp=0.08, prior_byz=prior_val,
        )
        elapsed = time.time() - t0
        all_results.append({
            "experiment": "exp3_prior", "method": "bst_gm",
            "byz_ratio": CFG["BYZ_RATIO"], "seed": seed,
            "param_name": "prior_byz", "param_value": prior_val,
            "asr": metrics["asr"], "fpr": metrics["fpr"],
            "balanced_accuracy": metrics["balanced_accuracy"],
            "tn": metrics["tn"], "fp": metrics["fp"],
            "fn": metrics["fn"], "tp": metrics["tp"],
            "elapsed_s": round(elapsed, 1),
        })
        gc.collect()
        print(_log_run(run_idx, n_runs_session1, "prior_byz", f"{prior_val:.2f}",
                       seed, metrics))
    _save_checkpoint(f"Exp3 prior={prior_val:.2f} complete")
 
print(f"\n  Exp3 complete. Per-prior summary:")
_exp3 = pd.DataFrame([r for r in all_results if r["experiment"]=="exp3_prior"])
for v in EXP3_PRIOR_VALS:
    s  = _exp3[_exp3["param_value"]==v]
    lbl = "oracle" if abs(v-0.40)<0.01 else ("uninformed" if v<0.30 else "max-entropy")
    print(f"    prior={v:.2f} ({lbl}): ASR={s['asr'].mean():.4f}%±{s['asr'].std():.4f}%  "
          f"max={s['asr'].max():.4f}%")
 
 
# ── Experiment 4: Ablation — BST-FedAvg ───────────────────────────────────────
print(f"\n{'─'*50}")
print("EXPERIMENT 4: Ablation — BST-FedAvg (pure trust weights, no GM)")
print("  Purpose: isolate the trust scoring contribution from the GM aggregation")
print("  [FIX-2] Uses τᵢ/Στ weights only — no dataset-size mixing")
print("  If BST-FedAvg < GM: trust scoring alone is effective")
print("  If BST-GM < BST-FedAvg: GM provides additional protection beyond trust")
print(f"{'─'*50}")
 
for seed in CFG["SEEDS"]:
    run_idx += 1
    t0 = time.time()
    metrics, byz_ids, n_byz = run_bst_experiment(
        method="bst_fedavg", seed=seed,
        p_d=0.70, p_fp=0.08, prior_byz=0.40,
    )
    elapsed = time.time() - t0
    all_results.append({
        "experiment": "exp4_ablation", "method": "bst_fedavg",
        "byz_ratio": CFG["BYZ_RATIO"], "seed": seed,
        "param_name": "ablation", "param_value": 0.0,
        "asr": metrics["asr"], "fpr": metrics["fpr"],
        "balanced_accuracy": metrics["balanced_accuracy"],
        "tn": metrics["tn"], "fp": metrics["fp"],
        "fn": metrics["fn"], "tp": metrics["tp"],
        "elapsed_s": round(elapsed, 1),
    })
    gc.collect()
    print(f"  seed={seed}: ASR={metrics['asr']:.4f}%  FPR={metrics['fpr']:.4f}%")
 
_save_checkpoint("Exp4 ablation complete")
 
_exp4 = pd.DataFrame([r for r in all_results if r["experiment"]=="exp4_ablation"])
print(f"\n  BST-FedAvg: ASR={_exp4['asr'].mean():.4f}%±{_exp4['asr'].std():.4f}%  "
      f"max={_exp4['asr'].max():.4f}%")
print(f"  (Ref) BST-GM:   ASR=0.0249%±0.0045%  max=0.0323%")
print(f"  (Ref) Plain GM: ASR=1.9658%±2.7316%  max=7.8442%")
 
 
# ── Final save ────────────────────────────────────────────────────────────────
df = pd.DataFrame(all_results)
csv_path = f"{CFG['OUT_DIR']}/{CFG['CSV_NAME']}"
df.to_csv(csv_path, index=False)
# Remove checkpoint (superseded by final)
if Path(_chk_path).exists():
    Path(_chk_path).unlink()
total_h = (time.time() - t_start) / 3600
print(f"\n{'='*65}")
print(f"SESSION 1 COMPLETE in {total_h:.2f} hours")
print(f"Total runs: {run_idx}")
print(f"Saved: {csv_path}  ({len(df)} rows)")
print("✓ Cell 9 complete")

SESSION 1: BST-GM SENSITIVITY EXPERIMENTS

──────────────────────────────────────────────────
EXPERIMENT 1: p_d sensitivity (CRITICAL)
  Fixed: p_fp=0.08, prior_byz=0.40, byz=40%
  Tests: deployability range — at what p_d does BST-GM remain robust?
──────────────────────────────────────────────────
  [  1/117] p_d=0.30  seed=  42  ASR=0.0291%  FPR=0.4489%  ETA=6.1h
  [  2/117] p_d=0.30  seed= 123  ASR=0.0242%  FPR=0.7901%  ETA=5.9h
  [  3/117] p_d=0.30  seed= 456  ASR=0.0355%  FPR=1.2390%  ETA=5.7h
  [  4/117] p_d=0.30  seed= 789  ASR=0.0420%  FPR=0.4669%  ETA=5.7h
  [  5/117] p_d=0.30  seed=1011  ASR=0.0307%  FPR=0.3771%  ETA=5.6h
  [  6/117] p_d=0.30  seed=1415  ASR=0.0226%  FPR=0.9158%  ETA=5.5h
  [  7/117] p_d=0.30  seed=1617  ASR=0.0452%  FPR=5.3870%  ETA=5.5h
  [  8/117] p_d=0.30  seed=1819  ASR=0.0210%  FPR=2.0111%  ETA=5.4h
  [  9/117] p_d=0.30  seed=2021  ASR=0.0194%  FPR=2.0111%  ETA=5.4h
  ✓ Checkpoint [Exp1 p_d=0.30 complete]: 9 rows → /kaggle/working/sensitivity_session1_c

In [12]:
# ===== CELL 10: Analysis — All Experiments =====
 
print("\n" + "="*65)
print("SESSION 1 ANALYSIS")
print("="*65)
 
# Exp1: p_d analysis
print("\nExperiment 1: p_d Sensitivity — Deployability Analysis")
print(f"{'p_d':>6}  {'Mean ASR':>10}  {'Std ASR':>10}  {'Max ASR':>10}  "
      f"{'Mean FPR':>10}  {'9/9 < 1%?':>10}")
print("-" * 65)
for v in sorted(df[df["experiment"]=="exp1_pd"]["param_value"].unique()):
    s = df[(df["experiment"]=="exp1_pd")&(df["param_value"]==v)]
    n_ok = (s["asr"] < 1.0).sum()
    print(f"  {v:.2f}  {s['asr'].mean():>10.4f}%  {s['asr'].std():>10.4f}%  "
          f"{s['asr'].max():>10.4f}%  {s['fpr'].mean():>10.4f}%  "
          f"{'YES ✓' if n_ok==9 else f'NO ({n_ok}/9)':>10}")
 
# Determine threshold
print()
for v in sorted(df[df["experiment"]=="exp1_pd"]["param_value"].unique()):
    s = df[(df["experiment"]=="exp1_pd")&(df["param_value"]==v)]
    if s["asr"].max() < 1.0:
        print(f"  Threshold: ALL seeds ASR < 1% for p_d >= {v:.2f}")
        break
 
# Exp2: p_fp analysis
print(f"\nExperiment 2: p_fp Sensitivity")
print(f"{'p_fp':>6}  {'Mean ASR':>10}  {'Mean FPR':>10}  {'Max ASR':>10}")
print("-" * 45)
for v in sorted(df[df["experiment"]=="exp2_pfp"]["param_value"].unique()):
    s = df[(df["experiment"]=="exp2_pfp")&(df["param_value"]==v)]
    print(f"  {v:.2f}  {s['asr'].mean():>10.4f}%  {s['fpr'].mean():>10.4f}%  "
          f"{s['asr'].max():>10.4f}%")
 
# Exp3: prior analysis
print(f"\nExperiment 3: prior_byz Sensitivity")
for v in sorted(df[df["experiment"]=="exp3_prior"]["param_value"].unique()):
    s = df[(df["experiment"]=="exp3_prior")&(df["param_value"]==v)]
    lbl = "oracle" if abs(v-0.40)<0.01 else ("uninformed" if v<0.30 else "max-entropy")
    print(f"  prior={v:.2f} ({lbl}): ASR={s['asr'].mean():.4f}%  max={s['asr'].max():.4f}%")
 
# Exp4: ablation
print(f"\nExperiment 4: Ablation Summary")
_e4 = df[df["experiment"]=="exp4_ablation"]
print(f"  FedAvg     (no trust, no GM):  16.5607% ± 21.6888%  max=57.3106%  [reference]")
print(f"  BST-FedAvg (trust, no GM):     {_e4['asr'].mean():.4f}% ± {_e4['asr'].std():.4f}%  max={_e4['asr'].max():.4f}%")
print(f"  Plain GM   (GM, no trust):      1.9658% ±  2.7316%  max= 7.8442%  [reference]")
print(f"  BST-GM     (trust + GM):        0.0249% ±  0.0045%  max= 0.0323%  [reference]")
 
if _e4['asr'].mean() < 1.9658:
    print(f"\n  → Trust scoring alone outperforms plain GM by "
          f"{1.9658 - _e4['asr'].mean():.4f} pp (mean ASR)")
    print(f"  → GM further reduces ASR by {_e4['asr'].mean() - 0.0249:.4f} pp")
    print(f"  VERDICT: Both trust AND GM components are effective.")
else:
    print(f"\n  → Trust scoring alone does NOT outperform plain GM.")
    print(f"  VERDICT: The GM aggregation is the primary contributor.")
 
print("✓ Cell 10 complete")


SESSION 1 ANALYSIS

Experiment 1: p_d Sensitivity — Deployability Analysis
   p_d    Mean ASR     Std ASR     Max ASR    Mean FPR   9/9 < 1%?
-----------------------------------------------------------------
  0.30      0.0300%      0.0093%      0.0452%      1.5163%       YES ✓
  0.40      0.0237%      0.0035%      0.0274%      1.2969%       YES ✓
  0.50      0.0226%      0.0035%      0.0274%      1.0096%       YES ✓
  0.60      0.0235%      0.0040%      0.0291%      1.0096%       YES ✓
  0.70      0.0249%      0.0045%      0.0323%      0.9976%       YES ✓

  Threshold: ALL seeds ASR < 1% for p_d >= 0.30

Experiment 2: p_fp Sensitivity
  p_fp    Mean ASR    Mean FPR     Max ASR
---------------------------------------------
  0.04      0.0244%      0.9697%      0.0307%
  0.08      0.0249%      0.9976%      0.0323%
  0.12      0.0246%      0.9896%      0.0323%
  0.15      0.0246%      0.9677%      0.0339%

Experiment 3: prior_byz Sensitivity
  prior=0.20 (uninformed): ASR=0.0249%  max=0

In [13]:
# ===== CELL 11: Statistical Tests (Holm-Bonferroni) =====
# [FIX-5] All p-values recomputed from the actual data in this notebook.
# [FIX-6] Reference per-seed values loaded from TRUST_GM_40PCT_PER_SEED dict
#         (verified from bst_gm_results.csv), not hardcoded inline.
 
if HAVE_SCIPY:
    print("\n" + "="*65)
    print("STATISTICAL TESTS — HOLM-BONFERRONI CORRECTED")
    print("="*65)
 
    seeds = CFG["SEEDS"]
    tgm_vals = [TRUST_GM_40PCT_PER_SEED[s] for s in seeds]  # [FIX-6]
    gm_vals  = [GM_40PCT_PER_SEED[s]       for s in seeds]  # [FIX-6]
 
    all_pvals = []
 
    # Test 1: BST-GM (p_d=0.70) vs Trust-GM — from prior work
    # Recomputed here for consistency
    bst_ref = [BST_GM_40PCT_PER_SEED[s] for s in seeds]
    try:
        stat, p = sp_stats.wilcoxon(bst_ref, tgm_vals, alternative="less")
        p = p if np.isfinite(p) else 1.0
    except: p = 1.0
    all_pvals.append(("BST-GM(p_d=0.70) vs Trust-GM", p))
 
    # Test 2-6: p_d sweep BST-GM vs Trust-GM
    for pd_val in EXP1_PD_VALS:
        sub = df[(df["experiment"]=="exp1_pd") & (df["param_value"]==pd_val)]
        if len(sub) == 9:
            bst_v = [sub[sub["seed"]==s]["asr"].values[0] for s in seeds]
            try:
                stat, p = sp_stats.wilcoxon(bst_v, tgm_vals, alternative="less")
                p = p if np.isfinite(p) else 1.0
            except: p = 1.0
            all_pvals.append((f"BST-GM(p_d={pd_val:.2f}) vs Trust-GM", p))
 
    # Test 7-9: prior sweep vs Trust-GM
    for prior_val in EXP3_PRIOR_VALS:
        sub = df[(df["experiment"]=="exp3_prior") & (df["param_value"]==prior_val)]
        if len(sub) == 9:
            bst_v = [sub[sub["seed"]==s]["asr"].values[0] for s in seeds]
            try:
                stat, p = sp_stats.wilcoxon(bst_v, tgm_vals, alternative="less")
                p = p if np.isfinite(p) else 1.0
            except: p = 1.0
            lbl = "oracle" if abs(prior_val-0.40)<0.01 else f"{prior_val:.2f}"
            all_pvals.append((f"BST-GM(prior={lbl}) vs Trust-GM", p))
 
    # Test 10: BST-FedAvg vs Plain GM (ablation significance)
    _e4_vals = [df[(df["experiment"]=="exp4_ablation")&(df["seed"]==s)]["asr"].values[0]
                for s in seeds]
    try:
        stat, p = sp_stats.wilcoxon(_e4_vals, gm_vals, alternative="less")
        p = p if np.isfinite(p) else 1.0
    except: p = 1.0
    all_pvals.append(("BST-FedAvg vs Plain GM", p))
 
    # Apply Holm-Bonferroni
    all_pvals_sorted = sorted(all_pvals, key=lambda x: x[1])
    m     = len(all_pvals_sorted)
    alpha = 0.05
    print(f"\n  m = {m} simultaneous tests")
    print(f"  {'Rank':<4} {'Comparison':<42} {'p':>9}  {'Threshold':>10}  {'Result'}")
    print("  " + "-" * 73)
    all_pass = True
    for i, (name, p) in enumerate(all_pvals_sorted, 1):
        threshold = alpha / (m - i + 1)
        result = "PASS ✓" if p < threshold else "FAIL ✗"
        if p >= threshold: all_pass = False
        print(f"  {i:<4} {name:<42} {p:>9.5f}  {threshold:>10.5f}  {result}")
    print(f"\n  All tests survive Holm-Bonferroni: {'YES ✓' if all_pass else 'NO ✗'}")
else:
    print("scipy not available — skipping statistical tests")
 
print("✓ Cell 11 complete")
 


STATISTICAL TESTS — HOLM-BONFERRONI CORRECTED

  m = 10 simultaneous tests
  Rank Comparison                                         p   Threshold  Result
  -------------------------------------------------------------------------
  1    BST-FedAvg vs Plain GM                       0.00195     0.00500  PASS ✓
  2    BST-GM(p_d=0.70) vs Trust-GM                 0.00977     0.00556  FAIL ✗
  3    BST-GM(p_d=0.40) vs Trust-GM                 0.00977     0.00625  FAIL ✗
  4    BST-GM(p_d=0.50) vs Trust-GM                 0.00977     0.00714  FAIL ✗
  5    BST-GM(p_d=0.60) vs Trust-GM                 0.00977     0.00833  FAIL ✗
  6    BST-GM(p_d=0.70) vs Trust-GM                 0.00977     0.01000  PASS ✓
  7    BST-GM(prior=0.20) vs Trust-GM               0.00977     0.01250  PASS ✓
  8    BST-GM(prior=oracle) vs Trust-GM             0.00977     0.01667  PASS ✓
  9    BST-GM(prior=0.50) vs Trust-GM               0.00977     0.02500  PASS ✓
  10   BST-GM(p_d=0.30) vs Trust-GM             

In [14]:
# ===== CELL 12: Save Final CSV =====
 
final_csv = f"{CFG['OUT_DIR']}/{CFG['CSV_NAME']}"
df.to_csv(final_csv, index=False)
print(f"Final results: {final_csv}  ({len(df)} rows)")
print(f"Shape: {df.shape}")
print(f"Experiments: {df['experiment'].value_counts().to_dict()}")
print()
print("REQUIRED FOR SESSION 2:")
print(f"  Download '{final_csv}' and upload as dataset to Session 2 Kaggle notebook.")
print("  Session 2 will load this file to create combined analysis.")
print("✓ Cell 12 complete")

Final results: /kaggle/working/sensitivity_session1.csv  (117 rows)
Shape: (117, 14)
Experiments: {'exp1_pd': 45, 'exp2_pfp': 36, 'exp3_prior': 27, 'exp4_ablation': 9}

REQUIRED FOR SESSION 2:
  Download '/kaggle/working/sensitivity_session1.csv' and upload as dataset to Session 2 Kaggle notebook.
  Session 2 will load this file to create combined analysis.
✓ Cell 12 complete


In [15]:
# ===== CELL 13: Visualization (Session 1) =====
 
if HAVE_PLOT:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle("Session 1: BST-GM Sensitivity Analysis (40% Byzantine)",
                 fontsize=13, fontweight="bold")
 
    # Plot 1: p_d sensitivity
    ax = axes[0, 0]
    _exp1 = df[df["experiment"]=="exp1_pd"]
    pd_vals = sorted(_exp1["param_value"].unique())
    means = [_exp1[_exp1["param_value"]==v]["asr"].mean() for v in pd_vals]
    stds  = [_exp1[_exp1["param_value"]==v]["asr"].std() for v in pd_vals]
    ax.errorbar(pd_vals, means, yerr=stds, marker="o", color="#8E24AA",
                lw=2, capsize=5, label="BST-GM")
    ax.axhline(y=0.0728, color="#43A047", ls="--", lw=1.5, label="Trust-GM (p_d=0.70)")
    ax.axhline(y=1.9658, color="#1E88E5", ls=":",  lw=1.5, label="Plain GM")
    ax.set_xlabel("Detection probability p_d"); ax.set_ylabel("Mean ASR (%)")
    ax.set_title("Exp 1: p_d Sensitivity")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
 
    # Plot 2: p_fp sensitivity (dual y-axis)
    ax = axes[0, 1]
    _exp2 = df[df["experiment"]=="exp2_pfp"]
    pfp_vals = sorted(_exp2["param_value"].unique())
    m_asr = [_exp2[_exp2["param_value"]==v]["asr"].mean() for v in pfp_vals]
    m_fpr = [_exp2[_exp2["param_value"]==v]["fpr"].mean() for v in pfp_vals]
    ax2   = ax.twinx()
    ax.plot(pfp_vals, m_asr, marker="o", color="#8E24AA", lw=2, label="Mean ASR (L)")
    ax2.plot(pfp_vals, m_fpr, marker="s", color="#FB8C00", ls="--", lw=2, label="Mean FPR (R)")
    ax.set_xlabel("False-positive rate p_fp")
    ax.set_ylabel("Mean ASR (%)", color="#8E24AA")
    ax2.set_ylabel("Mean FPR (%)", color="#FB8C00")
    ax.set_title("Exp 2: p_fp Sensitivity")
    ax.grid(True, alpha=0.3)
 
    # Plot 3: prior_byz sensitivity
    ax = axes[1, 0]
    _exp3 = df[df["experiment"]=="exp3_prior"]
    pv_vals = sorted(_exp3["param_value"].unique())
    means_p = [_exp3[_exp3["param_value"]==v]["asr"].mean() for v in pv_vals]
    stds_p  = [_exp3[_exp3["param_value"]==v]["asr"].std()  for v in pv_vals]
    ax.bar([f"{v:.2f}" for v in pv_vals], means_p, yerr=stds_p,
           color="#8E24AA", alpha=0.7, capsize=5)
    ax.axhline(y=0.0728, color="#43A047", ls="--", lw=1.5, label="Trust-GM")
    ax.set_xlabel("prior_byz"); ax.set_ylabel("Mean ASR (%)")
    ax.set_title("Exp 3: prior_byz Sensitivity")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3, axis="y")
 
    # Plot 4: Ablation bar chart
    ax = axes[1, 1]
    _exp4 = df[df["experiment"]=="exp4_ablation"]
    labels   = ["FedAvg\n(no trust, no GM)", "BST-FedAvg\n(trust only)",
                "GM\n(GM only)", "BST-GM\n(trust+GM)"]
    asr_vals = [16.5607, _exp4["asr"].mean(), 1.9658, 0.0249]
    std_vals = [21.6888, _exp4["asr"].std(),  2.7316, 0.0045]
    colors   = ["#E53935", "#FF9800", "#1E88E5", "#8E24AA"]
    bars = ax.bar(labels, asr_vals, yerr=std_vals, color=colors, alpha=0.8, capsize=5)
    ax.set_ylabel("Mean ASR (%) at 40% Byzantine")
    ax.set_title("Exp 4: Ablation Study")
    ax.grid(True, alpha=0.3, axis="y")
    for i, (v, s) in enumerate(zip(asr_vals, std_vals)):
        ax.text(i, v + s + 0.2, f"{v:.3f}%", ha="center", fontsize=8, fontweight="bold")
 
    plt.tight_layout()
    fig_path = f"{CFG['OUT_DIR']}/sensitivity_session1.png"
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Figure: {fig_path}")
 
print("=" * 65)
print("SESSION 1 COMPLETE")
print(f"CSV    : {final_csv}")
print(f"Runtime: {(time.time()-t_start)/3600:.2f} hours")
print()
print("NEXT STEP: Run Session 2 notebook for ACCCS-GM sensitivity (Exp 5 + 6)")
print("           Upload sensitivity_session1.csv as Kaggle dataset for Session 2")
print("=" * 65)
print("✓ Cell 13 complete — Session 1 done")
# This notebook is fully runnable on Kaggle without modification.

Figure: /kaggle/working/sensitivity_session1.png
SESSION 1 COMPLETE
CSV    : /kaggle/working/sensitivity_session1.csv
Runtime: 5.95 hours

NEXT STEP: Run Session 2 notebook for ACCCS-GM sensitivity (Exp 5 + 6)
           Upload sensitivity_session1.csv as Kaggle dataset for Session 2
✓ Cell 13 complete — Session 1 done
